In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table


# --------------------------------------------------
# User input
# --------------------------------------------------
ISO_FILE = Path(
    "/home/wyz5rge/SPISEA/evolution/Baraffe15/iso/z015/iso_6.00.fits"
)


# --------------------------------------------------
# Read and inspect the FITS file
# --------------------------------------------------
if not ISO_FILE.exists():
    raise FileNotFoundError(f"Isochrone file not found: {ISO_FILE}")

print(f"File: {ISO_FILE}")
print(f"Size: {ISO_FILE.stat().st_size / 1024**2:.3f} MB")

with fits.open(ISO_FILE, memmap=False) as hdul:
    print("\nHDU summary:")
    hdul.info()

    # Print basic information for every HDU.
    for hdu_index, hdu in enumerate(hdul):
        print("\n" + "=" * 80)
        print(f"HDU {hdu_index}")
        print(f"Name: {hdu.name}")
        print(f"Type: {type(hdu).__name__}")
        print(f"Data shape: {getattr(hdu.data, 'shape', None)}")

        # Show header without flooding the notebook with blank/comment cards.
        print("\nHeader:")
        for key, value in hdu.header.items():
            if key and key not in {"COMMENT", "HISTORY"}:
                print(f"  {key:20s} = {value}")

        if isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)):
            print("\nTable columns:")
            for column in hdu.columns:
                print(
                    f"  {column.name:25s} "
                    f"format={column.format!s:10s} "
                    f"unit={column.unit}"
                )

    # Locate the first table HDU automatically.
    table_hdu_indices = [
        i for i, hdu in enumerate(hdul)
        if isinstance(hdu, (fits.BinTableHDU, fits.TableHDU))
    ]

    if not table_hdu_indices:
        raise ValueError(
            "No binary-table or ASCII-table extension was found in this FITS file."
        )

    TABLE_HDU = table_hdu_indices[0]
    iso_table = Table(hdul[TABLE_HDU].data)

print(f"\nLoaded table from HDU {TABLE_HDU}.")
print(f"Rows: {len(iso_table)}")
print(f"Columns: {iso_table.colnames}")

# Astropy preview
display(iso_table[:10])

# Optional pandas representation for easier filtering/editing.
# Converting through a native ndarray avoids FITS big-endian dtype issues.
iso_df = iso_table.to_pandas()

print("\nPandas data types:")
display(iso_df.dtypes.to_frame("dtype"))

print("\nFirst 10 rows:")
display(iso_df.head(10))

File: /home/wyz5rge/SPISEA/evolution/Baraffe15/iso/z015/iso_6.00.fits
Size: 0.008 MB

HDU summary:
Filename: /home/wyz5rge/SPISEA/evolution/Baraffe15/iso/z015/iso_6.00.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     18   30R x 5C   [D, D, D, D, D]   

HDU 0
Name: PRIMARY
Type: PrimaryHDU
Data shape: None

Header:
  SIMPLE               = True
  BITPIX               = 8
  NAXIS                = 0
  EXTEND               = True

HDU 1
Name: 
Type: BinTableHDU
Data shape: (30,)

Header:
  XTENSION             = BINTABLE
  BITPIX               = 8
  NAXIS                = 2
  NAXIS1               = 40
  NAXIS2               = 30
  PCOUNT               = 0
  GCOUNT               = 1
  TFIELDS              = 5
  TTYPE1               = Mass
  TFORM1               = D
  TTYPE2               = Teff
  TFORM2               = D
  TTYPE3               = logL
  TFORM3               = D
  TTYPE4 

Mass,Teff,logL,logG,Rad
float64,float64,float64,float64,float64
0.01,2346.4805321446,-2.6965584035661996,3.5696688026746495,0.2721103991084502
0.015,2503.945268285694,-2.4196020488573633,3.5815473171430576,0.32783580485708275
0.02,2598.329318407256,-2.2427068159274435,3.594036134334699,0.3733172736290227
...,...,...,...,...
0.07,2895.949856335155,-1.4119498563351551,3.495424784502733,0.7825752154972674
0.072,2894.9024852969064,-1.389451242648453,3.4843537279453596,0.8040975147030937
0.075,2896.513176947581,-1.3600263538951625,3.4740263538951623,0.8304604691572564



Pandas data types:


,dtype
Mass,float64
Teff,float64
logL,float64
logG,float64
Rad,float64



First 10 rows:


,Mass,Teff,logL,logG,Rad
0,0.010,2346.480532,-2.696558,3.569669,0.272110
1,0.015,2503.945268,-2.419602,3.581547,0.327836
2,0.020,2598.329318,-2.242707,3.594036,0.373317
3,0.030,2709.964736,-1.971317,3.571317,0.469824
4,0.040,2779.000000,-1.811106,3.580106,0.536536
5,0.050,2823.562439,-1.643062,3.537062,0.631063
6,0.060,2864.000000,-1.524802,3.522802,0.702149
7,0.070,2895.949856,-1.411950,3.495425,0.782575
8,0.072,2894.902485,-1.389451,3.484354,0.804098
9,0.075,2896.513177,-1.360026,3.474026,0.830460
